# QM9 Property Prediction — Classifier Training
Trains an EGNN model to predict `relative_atomic_energy` on QM9.

In [ ]:
import sys, os, glob, re
import json, pickle

# Run from repo root; adjust paths to match main_qm9_prop.py's expectations
REPO_ROOT = os.path.abspath('.')
sys.path.insert(0, REPO_ROOT)
os.chdir(os.path.join(REPO_ROOT, 'qm9', 'property_prediction'))

import torch
from torch import nn, optim
import wandb

from qm9.property_prediction.models_property import EGNN, Naive, NumNodes
from qm9.property_prediction import prop_utils
from qm9 import dataset, utils

In [ ]:
# ── Configuration (mirrors SLURM args) ────────────────────────────────────────
import types
args = types.SimpleNamespace(
    exp_name           = 'exp_class_relenergy',
    property           = 'relative_atomic_energy',
    model_name         = 'egnn',
    lr                 = 5e-4,
    nf                 = 128,
    n_layers           = 7,
    attention          = 1,
    node_attr          = 0,
    batch_size         = 96,
    epochs             = 1000,
    num_workers        = 2,
    log_interval       = 20,
    test_interval      = 1,
    checkpoint_interval= 10,
    outf               = 'outputs',
    datadir            = '../../qm9/temp',
    dataset            = 'qm9_first_half',
    filter_n_atoms     = None,
    charge_power       = 2,
    remove_h           = False,
    include_charges    = True,
    weight_decay       = 1e-16,
    save_model         = True,
    resume             = 'latest',   # set to None or a path to override
    no_cuda            = False,
)

args.cuda   = not args.no_cuda and torch.cuda.is_available()
args.device = torch.device('cuda' if args.cuda else 'cpu')
print(args)

In [ ]:
# ── WandB + output directories ────────────────────────────────────────────────
os.environ.setdefault('WANDB_MODE', 'offline')
wandb.init(project='qm9_property_prediction', name=args.exp_name, config=vars(args))

prop_utils.makedir(args.outf)
prop_utils.makedir(args.outf + '/' + args.exp_name)

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────
dataloaders, charge_scale = dataset.retrieve_dataloaders(args)

# Use qm9_second_half as test set (mirrors main_qm9_prop.py)
args.dataset = 'qm9_second_half'
dataloaders_aux, _ = dataset.retrieve_dataloaders(args)
dataloaders['test'] = dataloaders_aux['train']

if args.property == 'relative_atomic_energy':
    relenergy_weights = utils.fit_relenergy_baseline(dataloaders['train'].dataset)
    for split in ('train', 'valid', 'test'):
        if split in dataloaders:
            utils.add_relative_atomic_energy(dataloaders[split].dataset, relenergy_weights)

property_norms = utils.compute_mean_mad_from_dataloader(dataloaders['valid'], [args.property])
mean = property_norms[args.property]['mean']
mad  = property_norms[args.property]['mad']
print(f'mean={mean:.4f}  mad={mad:.4f}')

In [ ]:
# ── Model, optimizer, scheduler ───────────────────────────────────────────────
def get_model(args):
    if args.model_name == 'egnn':
        return EGNN(in_node_nf=5, in_edge_nf=0, hidden_nf=args.nf,
                    device=args.device, n_layers=args.n_layers,
                    coords_weight=1.0, attention=args.attention,
                    node_attr=args.node_attr)
    elif args.model_name == 'naive':
        return Naive(device=args.device)
    elif args.model_name == 'numnodes':
        return NumNodes(device=args.device)
    raise ValueError(f'Unknown model: {args.model_name}')

# Resolve checkpoint to resume from
start_epoch = 0
if args.resume == 'latest':
    exp_dir = args.outf + '/' + args.exp_name
    ckpts   = glob.glob(exp_dir + '/checkpoint_epoch_*.npy')
    if ckpts:
        args.resume = max(ckpts, key=lambda p: int(re.search(r'checkpoint_epoch_(\d+)', p).group(1)))
        start_epoch = int(re.search(r'checkpoint_epoch_(\d+)', args.resume).group(1))
        print('Resuming from', args.resume)
    else:
        print('No checkpoints found in', exp_dir, '— starting from scratch')
        args.resume = None

model = get_model(args)
if args.resume is not None:
    model.load_state_dict(torch.load(args.resume, map_location=args.device))
    print('Loaded checkpoint from', args.resume)

optimizer    = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)
scaler       = torch.cuda.amp.GradScaler(enabled=args.cuda)
print(model)

In [ ]:
# ── Train / eval helpers ──────────────────────────────────────────────────────
loss_l1 = nn.L1Loss()

def run_epoch(model, epoch, loader, mean, mad, property, device,
              partition='train', optimizer=None, lr_scheduler=None,
              log_interval=20, scaler=None):
    if partition == 'train':
        lr_scheduler.step()
    res = {'loss': 0, 'counter': 0}
    loss_buf = []

    for i, data in enumerate(loader):
        batch_size, n_nodes, _ = data['positions'].size()
        atom_positions = data['positions'].view(batch_size * n_nodes, -1).to(device, torch.float32)
        atom_mask      = data['atom_mask'].view(batch_size * n_nodes, -1).to(device, torch.float32)
        edge_mask      = data['edge_mask'].view(batch_size * n_nodes * n_nodes, 1).to(device, torch.float32)
        nodes          = data['one_hot'].to(device, torch.float32).view(batch_size * n_nodes, -1)
        edges          = prop_utils.get_adj_matrix(n_nodes, batch_size, device)
        label          = data[property].to(device, torch.float32)

        if partition == 'train':
            model.train()
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                pred = model(h0=nodes, x=atom_positions, edges=edges, edge_attr=None,
                             node_mask=atom_mask, edge_mask=edge_mask, n_nodes=n_nodes)
                loss = loss_l1(pred, (label - mean) / mad)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            model.eval()
            with torch.cuda.amp.autocast():
                pred = model(h0=nodes, x=atom_positions, edges=edges, edge_attr=None,
                             node_mask=atom_mask, edge_mask=edge_mask, n_nodes=n_nodes)
            loss = loss_l1(mad * pred + mean, label)

        res['loss']    += loss.item() * batch_size
        res['counter'] += batch_size
        loss_buf.append(loss.item())

        if i % log_interval == 0:
            prefix = '' if partition == 'train' else f'>> {partition}\t'
            window = loss_buf[-10:]
            print(f"{prefix}Epoch {epoch}\tIter {i}\tloss {sum(window)/len(window):.4f}")

    return res['loss'] / res['counter']

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
res = {'epochs': [], 'losses': [], 'best_val': 1e10, 'best_test': 1e10, 'best_epoch': 0}

for epoch in range(start_epoch, args.epochs):
    train_loss = run_epoch(model, epoch, dataloaders['train'], mean, mad, args.property,
                           args.device, partition='train', optimizer=optimizer,
                           lr_scheduler=lr_scheduler, log_interval=args.log_interval,
                           scaler=scaler)

    if epoch % args.test_interval == 0:
        val_loss  = run_epoch(model, epoch, dataloaders['valid'], mean, mad, args.property,
                              args.device, partition='valid', optimizer=optimizer,
                              lr_scheduler=lr_scheduler, log_interval=args.log_interval,
                              scaler=scaler)
        test_loss = run_epoch(model, epoch, dataloaders['test'],  mean, mad, args.property,
                              args.device, partition='test',  log_interval=args.log_interval,
                              scaler=scaler)
        res['epochs'].append(epoch)
        res['losses'].append(test_loss)

        if val_loss < res['best_val']:
            res['best_val']   = val_loss
            res['best_test']  = test_loss
            res['best_epoch'] = epoch
            if args.save_model:
                torch.save(model.state_dict(),
                           f"{args.outf}/{args.exp_name}/best_checkpoint.npy")
                with open(f"{args.outf}/{args.exp_name}/args.pickle", 'wb') as f:
                    pickle.dump(args, f)

        print(f"Val {val_loss:.4f}  Test {test_loss:.4f}  Epoch {epoch}")
        print(f"Best — Val {res['best_val']:.4f}  Test {res['best_test']:.4f}  Epoch {res['best_epoch']}")
        wandb.log({'val_loss': val_loss, 'test_loss': test_loss,
                   'best_val': res['best_val'], 'best_test': res['best_test']}, step=epoch)

    wandb.log({'train_loss': train_loss, 'lr': lr_scheduler.get_last_lr()[0]}, step=epoch)

    if args.save_model and args.checkpoint_interval > 0 and epoch % args.checkpoint_interval == 0:
        torch.save(model.state_dict(),
                   f"{args.outf}/{args.exp_name}/checkpoint_epoch_{epoch}.npy")

    with open(f"{args.outf}/{args.exp_name}/losses.json", 'w') as f:
        json.dump(res, f, indent=4)

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
print(f"Best val loss:  {res['best_val']:.4f}")
print(f"Best test loss: {res['best_test']:.4f}")
print(f"Best epoch:     {res['best_epoch']}")
wandb.finish()